# Construção do painel horário do SIN — 2025

Execute este notebook **antes** de `06_implementacao_documento_mfg_v4.ipynb` ou `07_mfg_resumo.ipynb`. Ele lê os CSVs de `data/` por meio de `validate_model.pipeline` e grava os dois caches esperados pelos notebooks seguintes:

- `outputs/cache/panel_hourly_v4.parquet`
- `outputs/cache/cmo_hourly_v4.parquet`

A última célula relê os arquivos gravados e valida o contrato mínimo do painel.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "validate_model" / "pipeline.py").exists():
            return candidate
    raise FileNotFoundError("Raiz do projeto não encontrada (validate_model/pipeline.py).")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from validate_model.pipeline import SINPaths, build_panel, load_cmo_horario

YEAR = 2025
CACHE_DIR = ROOT / "outputs" / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

paths = SINPaths(root=str(ROOT), year=YEAR)
print(f"ROOT: {ROOT}")
paths.summary()

## 1. Painel físico

O painel tem uma linha por hora e subsistema. A construção centralizada em `build_panel` mantém as mesmas regras de leitura, normalização e agregação usadas pelos demais notebooks.

In [ ]:
if not paths.curva_paths:
    raise FileNotFoundError(f"Dados de carga não encontrados em {ROOT / 'data' / 'demanda_efetiva'}")

panel = build_panel(paths).copy()
panel["din_instante"] = pd.to_datetime(panel["din_instante"], errors="coerce")
start = pd.Timestamp(YEAR, 1, 1)
end = pd.Timestamp(YEAR + 1, 1, 1)
panel = panel[(panel["din_instante"] >= start) & (panel["din_instante"] < end)].copy()
panel = panel.sort_values(["id_subsistema", "din_instante"]).reset_index(drop=True)

required_columns = {
    "din_instante", "id_subsistema", "D", "x_int", "D_net",
    "gs", "gr", "gh", "g_nuc_obs", "g_th_obs", "gn_obs",
}
missing_columns = sorted(required_columns.difference(panel.columns))
if missing_columns:
    raise RuntimeError(f"Painel sem colunas obrigatórias: {missing_columns}")
if panel.empty:
    raise RuntimeError("O painel ficou vazio após o recorte de 2025.")
if panel[["din_instante", "id_subsistema"]].isna().any().any():
    raise RuntimeError("Há chaves nulas no painel.")
duplicates = int(panel.duplicated(["din_instante", "id_subsistema"]).sum())
if duplicates:
    raise RuntimeError(f"Há {duplicates} chaves hora × subsistema duplicadas.")

panel_path = CACHE_DIR / "panel_hourly_v4.parquet"
panel.to_parquet(panel_path, index=False)
print(f"Painel salvo: {panel_path}")
print(f"Linhas: {len(panel):,} | colunas: {panel.shape[1]}")
print(f"Período: {panel['din_instante'].min()} -> {panel['din_instante'].max()}")
display(panel.groupby("id_subsistema").agg(
    inicio=("din_instante", "min"),
    fim=("din_instante", "max"),
    horas=("din_instante", "size"),
    demanda_preenchida=("D", "count"),
))

## 2. CMO horário

O CMO semi-horário é agregado pela média das duas observações de cada hora e salvo separadamente para o *merge* feito nos notebooks de modelagem.

In [ ]:
cmo = load_cmo_horario(paths.cmo_semihorario_path).copy()
cmo["din_instante"] = pd.to_datetime(cmo["din_instante"], errors="coerce")
cmo = cmo[(cmo["din_instante"] >= start) & (cmo["din_instante"] < end)].copy()
cmo = cmo.sort_values(["id_subsistema", "din_instante"]).reset_index(drop=True)
if cmo.empty or cmo["cmo_h"].isna().all():
    raise RuntimeError("O cache horário de CMO ficou vazio.")
if cmo.duplicated(["din_instante", "id_subsistema"]).any():
    raise RuntimeError("Há chaves hora × subsistema duplicadas no CMO.")

cmo_path = CACHE_DIR / "cmo_hourly_v4.parquet"
cmo.to_parquet(cmo_path, index=False)

metadata = {
    "year": YEAR,
    "panel_rows": len(panel),
    "panel_columns": panel.columns.tolist(),
    "panel_start": panel["din_instante"].min().isoformat(),
    "panel_end": panel["din_instante"].max().isoformat(),
    "cmo_rows": len(cmo),
}
metadata_path = CACHE_DIR / "panel_hourly_v4.metadata.json"
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"CMO salvo: {cmo_path} ({len(cmo):,} linhas)")
print(f"Metadados: {metadata_path}")

## 3. Verificação dos artefatos

Esta célula valida os arquivos no disco, não apenas os DataFrames que ainda estão em memória.

In [ ]:
panel_check = pd.read_parquet(panel_path)
cmo_check = pd.read_parquet(cmo_path)
assert len(panel_check) == len(panel)
assert required_columns.issubset(panel_check.columns)
assert not panel_check.duplicated(["din_instante", "id_subsistema"]).any()
assert len(cmo_check) == len(cmo)
assert {"din_instante", "id_subsistema", "cmo_h"}.issubset(cmo_check.columns)
print("OK — caches do painel e do CMO prontos para os notebooks 06 e 07.")